In [1]:
# Importing modules
import pandas as pd
import numpy as np
from src.utils.smiles2morganfp import MorganFP
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Loading ESOL data
esol_data = pd.read_csv("data/processed/train/ESOL.csv")

# Loading Lipophilicity data
lipophilicity_data = pd.read_csv("data/processed/train/Lipophilicity.csv")

# Loading B3DB data
b3db_data = pd.read_csv("data/processed/train/B3DB.csv")

# Loading RT data
rt_data = pd.read_csv("data/processed/train/RT.csv")

In [3]:
# Function to run analysis
def RunML(data, dataName, modelName):

	# Generate FP
	fp = MorganFP(data["smiles"])
	fp["smiles"] = fp.index
	fp = fp.merge(data, on="smiles")
    
	# Feature Matrix (molecular fingerprint)
	X = fp.drop(["smiles", "target"], axis=1)

	# Target labels
	y = fp["target"]

	# Train test split
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

	# Model
	model = GridSearchCV(model_dict[modelName], params_dict[modelName], cv=None, scoring="neg_root_mean_squared_error")

	# Model training
	model = model.fit(X_train.to_numpy(), y_train)

	# Model testing
	y_pred = model.predict(X_test.to_numpy())

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)

	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
            "Model Params":[model.best_params_],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)]})

In [4]:
# Function for model selection and hyperparameter tuning
def search_hyperparams(dataName):
    temp_out = []
    # Hyperparameter tuning loop
    for modelName in model_dict.keys():
        temp_out.append(RunML(data_dict[dataName], dataName, modelName))

    # Saving results
    final_df = pd.concat(temp_out, ignore_index=True)
    final_df.to_csv(f"results/Output_Hyperparameter_Optimization_ML_{dataName}.csv", quoting=False, index=False)

    # Best parameters
    for model in model_dict.keys():
        temp = final_df[final_df["Model"]==model]
        print(temp.sort_values(by=["RMSE"]).head(1),"\n")

In [5]:
# Models
model_dict = {
    "LR": LinearRegression(), 
    "SVM": SVR(),
    "RF": RandomForestRegressor(random_state=42),
    "XGB": XGBRegressor(random_state=42, objective='reg:squarederror')
}

# Params
params_dict = {
    'LR': {
        'fit_intercept': [True],
        'positive': [False]
    },

    'SVM': {
        'kernel': ['rbf'],
        'C': [0.1, 1, 10],
        'gamma': ['scale'],
        'epsilon': [0.05]
    },

    'RF': {
        'n_estimators': [200],
        'max_depth': [None, 10],
        'min_samples_split': [4],
        'min_samples_leaf': [5],
        'max_features': ['sqrt']
    },

    'XGB': {
        'n_estimators': [200],
        'max_depth': [3, 6],
        'learning_rate': [0.05],
        'subsample': [0.8],
        'colsample_bytree': [0.8],
        'alpha': [0]
    }
}

# Data
data_dict = {
    "ESOL": esol_data, 
    "Lipophilicity": lipophilicity_data,
    "RT": rt_data,
    "B3DB": b3db_data
}

In [6]:
# Params Optim for ESOL
search_hyperparams("ESOL")

   Data Model                                Model Params  MAE  RMSE
0  ESOL    LR  {'fit_intercept': True, 'positive': False}  2.1  2.96 

   Data Model                                       Model Params   MAE  RMSE
1  ESOL   SVM  {'C': 10, 'epsilon': 0.05, 'gamma': 'scale', '...  0.89  1.29 

   Data Model                                       Model Params   MAE  RMSE
2  ESOL    RF  {'max_depth': None, 'max_features': 'sqrt', 'm...  1.29  1.71 

   Data Model                                       Model Params   MAE  RMSE
3  ESOL   XGB  {'alpha': 0, 'colsample_bytree': 0.8, 'learnin...  0.96  1.34 



In [7]:
# Params Optim for Lipophilicity
search_hyperparams("Lipophilicity")

            Data Model                                Model Params   MAE  RMSE
0  Lipophilicity    LR  {'fit_intercept': True, 'positive': False}  1.17  1.58 

            Data Model                                       Model Params  \
1  Lipophilicity   SVM  {'C': 10, 'epsilon': 0.05, 'gamma': 'scale', '...   

    MAE  RMSE  
1  0.71  0.91   

            Data Model                                       Model Params  \
2  Lipophilicity    RF  {'max_depth': None, 'max_features': 'sqrt', 'm...   

    MAE  RMSE  
2  0.86  1.08   

            Data Model                                       Model Params  \
3  Lipophilicity   XGB  {'alpha': 0, 'colsample_bytree': 0.8, 'learnin...   

    MAE  RMSE  
3  0.75  0.97   



In [8]:
# Params Optim for RT
search_hyperparams("RT")

  Data Model                                Model Params     MAE    RMSE
0   RT    LR  {'fit_intercept': True, 'positive': False}  137.29  188.02 

  Data Model                                       Model Params     MAE  \
1   RT   SVM  {'C': 10, 'epsilon': 0.05, 'gamma': 'scale', '...  129.19   

     RMSE  
1  167.54   

  Data Model                                       Model Params     MAE  \
2   RT    RF  {'max_depth': None, 'max_features': 'sqrt', 'm...  105.38   

     RMSE  
2  124.61   

  Data Model                                       Model Params   MAE    RMSE
3   RT   XGB  {'alpha': 0, 'colsample_bytree': 0.8, 'learnin...  73.8  102.83 



In [9]:
# Params Optim for B3DB
search_hyperparams("B3DB")

   Data Model                                Model Params   MAE  RMSE
0  B3DB    LR  {'fit_intercept': True, 'positive': False}  0.53  0.74 

   Data Model                                       Model Params   MAE  RMSE
1  B3DB   SVM  {'C': 1, 'epsilon': 0.05, 'gamma': 'scale', 'k...  0.38  0.53 

   Data Model                                       Model Params   MAE  RMSE
2  B3DB    RF  {'max_depth': None, 'max_features': 'sqrt', 'm...  0.47  0.61 

   Data Model                                       Model Params  MAE  RMSE
3  B3DB   XGB  {'alpha': 0, 'colsample_bytree': 0.8, 'learnin...  0.4  0.54 

